In [ ]:
import os
import shutil
import tensorflow as tf
from tqdm import tqdm

print("Num GPUs Available: ", len(tf.config.list_physical_devices("GPU")))

In [ ]:
dataset_path = "dataset_image"
output_dir = "dataset_centralcrop"
error_dir = os.path.join(output_dir, "_error_files")

os.makedirs(output_dir, exist_ok=True)
os.makedirs(error_dir, exist_ok=True)

IMG_SIZE = (256, 256)


def resize_and_crop_save(input_path, output_path, crop_fraction=0.85):
    try:
        img_raw = tf.io.read_file(input_path)
        img_tensor = tf.image.decode_image(img_raw, channels=3, expand_animations=False)
        img_resized = tf.image.resize(img_tensor, [300, 300])
        img_cropped = tf.image.central_crop(img_resized, central_fraction=crop_fraction)
        img_final = tf.image.resize(img_cropped, IMG_SIZE)
        img_final = tf.cast(img_final, tf.uint8)
        img_bytes = tf.io.encode_jpeg(img_final)
        tf.io.write_file(output_path, img_bytes)

    except Exception:
        try:
            shutil.copy(input_path, error_dir)
        except Exception:
            pass


for class_name in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, class_name)
    if not os.path.isdir(class_path):
        continue

    output_class_path = os.path.join(output_dir, class_name)
    os.makedirs(output_class_path, exist_ok=True)

    files = [
        f
        for f in os.listdir(class_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    for file_name in tqdm(files, desc=class_name, unit="img"):
        input_file = os.path.join(class_path, file_name)
        output_file = os.path.join(output_class_path, file_name)
        resize_and_crop_save(input_file, output_file)